# Day 09. Exercise 02
# Metrics

## 0. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv').reset_index()
df_for_merge = pd.read_csv('../data/dayofweek.csv').reset_index()
df['dayofweek'] = df_for_merge['dayofweek']

df = df.drop(columns='index', axis=1)

X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1348, 43), (338, 43), (1348,), (338,))

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [3]:
svc = SVC(
    C = 10, 
    class_weight = None, 
    gamma = 'auto', 
    kernel = 'rbf', 
    random_state = 21, 
    probability=True)

svc = svc.fit(X_train, y_train)

y_pred = svc.predict(X_test)
proba = svc.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, proba, multi_class='ovo', average='weighted')

print(f'accuracy = {accuracy}\nprecision = {precision}\nrecall = {recall}\nroc_auc = {roc_auc}')

accuracy = 0.8875739644970414
precision = 0.8926729169690374
recall = 0.8875739644970414
roc_auc = 0.9787793228216216


## 3. Decision tree

1. The same task for decision tree

In [4]:
dtc = DecisionTreeClassifier(
    class_weight = 'balanced',
    criterion = 'gini',
    max_depth = 22,
    random_state = 21)

dtc = dtc.fit(X_train, y_train)

y_pred = dtc.predict(X_test)
proba = dtc.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, proba, multi_class='ovo', average='weighted')

print(f'accuracy = {accuracy}\nprecision = {precision}\nrecall = {recall}\nroc_auc = {roc_auc}')

accuracy = 0.8905325443786982
precision = 0.8926192681313897
recall = 0.8905325443786982
roc_auc = 0.9366351447213223


## 4. Random forest

1. The same task for random forest.

In [5]:
rfc = RandomForestClassifier(
    class_weight = None,
    criterion = 'gini',
    max_depth = 28,
    n_estimators = 50,
    random_state = 21
    )

rfc = rfc.fit(X_train, y_train)

y_pred = rfc.predict(X_test)
proba = rfc.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, proba, multi_class='ovo', average='weighted')

print(f'accuracy = {accuracy}\nprecision = {precision}\nrecall = {recall}\nroc_auc = {roc_auc}')

accuracy = 0.9289940828402367
precision = 0.9300865038851309
recall = 0.9289940828402367
roc_auc = 0.9903274757720744


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [6]:
errors = y_test[y_test != y_pred]
total_per_day = y_test.value_counts()
errors_per_day = errors.value_counts()
error_rate_per_day = (errors_per_day / total_per_day).fillna(0) * 100

lab_columns = [col for col in df.columns if col.startswith("labname_")]
errors_labs = X_test.loc[errors.index, lab_columns].sum()
error_rate_per_lab = (errors_labs / len(X_test)) * 100

user_columns = [col for col in df.columns if col.startswith("uid_user_")]
errors_users = X_test.loc[errors.index, user_columns].sum()
error_rate_per_user = (errors_users / len(X_test)) * 100

print("Ошибка по дням недели (% от общего количества выборок для этого дня):")
print(error_rate_per_day.sort_values(ascending=False))

print("\nОшибка по лабораторным работам (% от общего количества тестовых данных):")
print(error_rate_per_lab.sort_values(ascending=False))

print("\nОшибка по пользователям (% от общего количества тестовых данных):")
print(error_rate_per_user.sort_values(ascending=False))

Ошибка по дням недели (% от общего количества выборок для этого дня):
dayofweek
0    25.925926
4    14.285714
1    10.909091
2     6.666667
5     5.555556
3     2.500000
6     1.408451
Name: count, dtype: float64

Ошибка по лабораторным работам (% от общего количества тестовых данных):
labname_project1    2.662722
labname_laba04      1.775148
labname_laba06      0.591716
labname_laba06s     0.591716
labname_code_rvw    0.295858
labname_lab03       0.295858
labname_lab03s      0.295858
labname_lab05s      0.295858
labname_laba05      0.295858
labname_lab02       0.000000
labname_laba04s     0.000000
dtype: float64

Ошибка по пользователям (% от общего количества тестовых данных):
uid_user_19    1.183432
uid_user_25    0.591716
uid_user_4     0.591716
uid_user_3     0.591716
uid_user_31    0.591716
uid_user_16    0.591716
uid_user_24    0.295858
uid_user_27    0.295858
uid_user_29    0.295858
uid_user_2     0.295858
uid_user_22    0.295858
uid_user_18    0.295858
uid_user_14    0.295858


In [7]:
rfc = RandomForestClassifier(n_estimators=100, max_depth=24, criterion='entropy', class_weight='balanced', random_state=21)
rfc.fit(X_train, y_train)
y_pred = rfc.predict(X_test)

df = pd.merge(X_test, y_test, left_index=True, right_index=True)
df['prediction'] = y_pred
df.head()

joblib.dump(rfc, '../data/model_02.pkl')

['../data/model_02.pkl']

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [8]:
def get_metrics(model):
    data = {}
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)
    data['accuracy'] = accuracy_score(y_test, y_pred)
    data['precision'] = precision_score(y_test, y_pred, average='weighted')
    data['recall'] = recall_score(y_test, y_pred, average='weighted')
    data['roc_auc'] = roc_auc_score(y_test, y_score, multi_class='ovo', average='weighted')
    return data

In [9]:
get_metrics(svc)

{'accuracy': 0.8875739644970414,
 'precision': 0.8926729169690374,
 'recall': 0.8875739644970414,
 'roc_auc': np.float64(0.9787793228216216)}

In [10]:
get_metrics(dtc)

{'accuracy': 0.8905325443786982,
 'precision': 0.8926192681313897,
 'recall': 0.8905325443786982,
 'roc_auc': np.float64(0.9366351447213223)}

In [11]:
get_metrics(rfc)

{'accuracy': 0.9319526627218935,
 'precision': 0.9340183677910212,
 'recall': 0.9319526627218935,
 'roc_auc': np.float64(0.9882077759075321)}